# Geospatial H3 Analytics

**Purpose**: Demonstrate Snowflake's native H3 geospatial capabilities for grid analysis

This notebook showcases:
- H3 hexagonal indexing for spatial aggregation
- Grid asset density analysis
- Vegetation risk corridor mapping
- Service territory visualization

**H3 Benefits**:
- Consistent hexagonal cells (no distortion at edges)
- Hierarchical resolution (zoom in/out)
- Efficient spatial joins and aggregations

---

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
import pandas as pd

session = get_active_session()
session.use_database("FLUX_DATABASE")
session.use_schema("PRODUCTION")

# Verify H3 functions available
test = session.sql("SELECT H3_LATLNG_TO_CELL(29.76, -95.36, 7) as H3_INDEX").collect()
print(f"H3 functions available: ✓")
print(f"Sample H3 index (Houston): {test[0][0]}")

## 1. H3 Indexing for Grid Assets

Convert lat/lon coordinates to H3 hexagonal cells

In [ ]:
# Index transformers to H3 resolution 8 (avg area ~0.74 km²)
transformer_h3 = session.sql("""
    SELECT 
        TRANSFORMER_ID,
        LATITUDE,
        LONGITUDE,
        H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 8) as H3_INDEX_R8,
        H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 7) as H3_INDEX_R7,
        H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 6) as H3_INDEX_R6,
        RATED_KVA,
        HEALTH_SCORE
    FROM TRANSFORMER_METADATA
    WHERE LATITUDE IS NOT NULL
    LIMIT 1000
""").to_pandas()

print(f"Indexed {len(transformer_h3)} transformers to H3")
print(f"Unique H3 cells (R8): {transformer_h3['H3_INDEX_R8'].nunique()}")
print(f"Unique H3 cells (R7): {transformer_h3['H3_INDEX_R7'].nunique()}")
transformer_h3.head()

## 2. Transformer Density Heatmap

Aggregate transformer count and capacity by H3 cell

In [ ]:
# Aggregate by H3 cell
density_map = session.sql("""
    SELECT 
        H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 7) as H3_INDEX,
        H3_CELL_TO_LAT(H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 7)) as CENTER_LAT,
        H3_CELL_TO_LNG(H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 7)) as CENTER_LON,
        COUNT(*) as TRANSFORMER_COUNT,
        SUM(RATED_KVA) as TOTAL_CAPACITY_KVA,
        ROUND(AVG(HEALTH_SCORE), 2) as AVG_HEALTH_SCORE,
        SUM(METER_COUNT) as TOTAL_METERS
    FROM TRANSFORMER_METADATA
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL
    GROUP BY 1, 2, 3
    HAVING TRANSFORMER_COUNT >= 5
    ORDER BY TRANSFORMER_COUNT DESC
    LIMIT 100
""").to_pandas()

print(f"Density map: {len(density_map)} H3 cells with 5+ transformers")
print(f"\nTop 10 densest areas:")
density_map.head(10)

## 3. H3 Hierarchical Rollup

Aggregate from fine (R8) to coarse (R5) resolution

In [ ]:
# Hierarchical aggregation
hierarchy = session.sql("""
    WITH base AS (
        SELECT 
            H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 8) as H3_R8,
            COUNT(*) as XFMR_COUNT,
            SUM(RATED_KVA) as CAPACITY
        FROM TRANSFORMER_METADATA
        WHERE LATITUDE IS NOT NULL
        GROUP BY 1
    )
    SELECT 
        'Resolution 8' as LEVEL,
        COUNT(DISTINCT H3_R8) as CELL_COUNT,
        SUM(XFMR_COUNT) as TOTAL_TRANSFORMERS,
        ROUND(AVG(XFMR_COUNT), 2) as AVG_PER_CELL
    FROM base
    UNION ALL
    SELECT 
        'Resolution 7',
        COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 7)),
        SUM(XFMR_COUNT),
        ROUND(SUM(XFMR_COUNT) / COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 7)), 2)
    FROM base
    UNION ALL
    SELECT 
        'Resolution 6',
        COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 6)),
        SUM(XFMR_COUNT),
        ROUND(SUM(XFMR_COUNT) / COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 6)), 2)
    FROM base
    UNION ALL
    SELECT 
        'Resolution 5',
        COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 5)),
        SUM(XFMR_COUNT),
        ROUND(SUM(XFMR_COUNT) / COUNT(DISTINCT H3_CELL_TO_PARENT(H3_R8, 5)), 2)
    FROM base
""").to_pandas()

print("H3 Hierarchical Aggregation:")
hierarchy

## 4. Spatial Join: Meters to Transformers

Use H3 for efficient spatial proximity matching

In [ ]:
# Spatial join using H3 (much faster than ST_DISTANCE)
spatial_join = session.sql("""
    WITH meter_h3 AS (
        SELECT 
            METER_ID,
            METER_LATITUDE,
            METER_LONGITUDE,
            H3_LATLNG_TO_CELL(METER_LATITUDE, METER_LONGITUDE, 9) as H3_INDEX
        FROM METER_INFRASTRUCTURE
        WHERE METER_LATITUDE IS NOT NULL
        LIMIT 10000
    ),
    transformer_h3 AS (
        SELECT 
            TRANSFORMER_ID,
            LATITUDE,
            LONGITUDE,
            H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 9) as H3_INDEX
        FROM TRANSFORMER_METADATA
        WHERE LATITUDE IS NOT NULL
    )
    SELECT 
        m.METER_ID,
        t.TRANSFORMER_ID,
        m.H3_INDEX,
        -- H3 grid distance (number of cells)
        H3_GRID_DISTANCE(m.H3_INDEX, t.H3_INDEX) as H3_DISTANCE
    FROM meter_h3 m
    JOIN transformer_h3 t ON m.H3_INDEX = t.H3_INDEX
    LIMIT 100
""").to_pandas()

print(f"Meters matched to transformers in same H3 cell: {len(spatial_join)}")
spatial_join.head()

## 5. K-Ring Neighbor Analysis

Find all assets within k hexagons of a point

In [ ]:
# Get all transformers within 2 hexagons of downtown Houston
downtown_lat, downtown_lon = 29.7604, -95.3698

kring_query = session.sql(f"""
    WITH downtown_cell AS (
        SELECT H3_LATLNG_TO_CELL({downtown_lat}, {downtown_lon}, 7) as CENTER_H3
    ),
    neighbor_cells AS (
        SELECT VALUE as H3_INDEX
        FROM downtown_cell,
        LATERAL FLATTEN(H3_GRID_DISK(CENTER_H3, 2))
    )
    SELECT 
        t.TRANSFORMER_ID,
        t.LATITUDE,
        t.LONGITUDE,
        t.RATED_KVA,
        t.HEALTH_SCORE,
        H3_LATLNG_TO_CELL(t.LATITUDE, t.LONGITUDE, 7) as H3_INDEX
    FROM TRANSFORMER_METADATA t
    JOIN neighbor_cells n ON H3_LATLNG_TO_CELL(t.LATITUDE, t.LONGITUDE, 7) = n.H3_INDEX
    WHERE t.LATITUDE IS NOT NULL
""").to_pandas()

print(f"Transformers within 2 hexagons of downtown Houston: {len(kring_query)}")
kring_query.head(10)

## 6. Vegetation Risk Corridor Mapping

Identify high-risk vegetation areas near power lines

In [ ]:
# Vegetation risk by H3 cell
veg_risk = session.sql("""
    SELECT 
        H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 8) as H3_INDEX,
        H3_CELL_TO_LAT(H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 8)) as CENTER_LAT,
        H3_CELL_TO_LNG(H3_LATLNG_TO_CELL(LATITUDE, LONGITUDE, 8)) as CENTER_LON,
        COUNT(*) as RISK_POINTS,
        ROUND(AVG(RISK_SCORE), 2) as AVG_RISK_SCORE,
        MAX(RISK_SCORE) as MAX_RISK_SCORE
    FROM VEGETATION_RISK_ENHANCED
    WHERE LATITUDE IS NOT NULL
    GROUP BY 1, 2, 3
    HAVING AVG_RISK_SCORE > 0.7
    ORDER BY AVG_RISK_SCORE DESC
    LIMIT 50
""").to_pandas()

print(f"High vegetation risk H3 cells: {len(veg_risk)}")
veg_risk.head(10)

## Key Takeaways

1. **H3 Indexing**: Native Snowflake functions for hexagonal spatial indexing
2. **Hierarchical Resolution**: Zoom from R5 (large area) to R9 (street level)
3. **Efficient Joins**: H3 cell matching is faster than distance calculations
4. **K-Ring Analysis**: Find all assets within k hexagons
5. **Aggregation**: Roll up point data to hexagonal cells for visualization

**H3 Resolution Reference**:
| Resolution | Avg Area | Use Case |
|------------|----------|----------|
| 5 | ~252 km² | Regional planning |
| 6 | ~36 km² | District analysis |
| 7 | ~5.2 km² | Service territory |
| 8 | ~0.74 km² | Neighborhood |
| 9 | ~0.11 km² | Block level |